<a href="https://colab.research.google.com/github/OscarPerilla/unicafam_Modelos_Analiticos_Indra/blob/main/Sesion3_Integracion_de_Datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Integración de datos**

In [14]:
# Importar librerías de análisis y StringIO para leer texto como archivo CSV en memoria
import pandas as pd
import numpy as np
from io import StringIO

In [15]:
csv_clientes = """id_cliente;nombre;ciudad;segmento
C001;Ana Ruiz;Bogotá;Oro
C002;Luis Peña;Medellín;Plata
C003;Marta Gil;Cali;Oro
C004;Juan Ríos;Bogotá;Bronce
C005;Sofía Mora;Barranquilla;Plata
C006;Carlos Puenes;Pasto;Bronce
c007;Patricia Mojica;Manizales;plata
C008;Juan Velez;Bucaramanga;Bronce"""

In [16]:
# Cargar el dataset de ejemplo de California Housing incluido en Colab
df_california_housing = pd.read_csv("/content/sample_data/california_housing_train.csv")

In [17]:
df_california_housing

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0
4,-114.57,33.57,20.0,1454.0,326.0,624.0,262.0,1.9250,65500.0
...,...,...,...,...,...,...,...,...,...
16995,-124.26,40.58,52.0,2217.0,394.0,907.0,369.0,2.3571,111400.0
16996,-124.27,40.69,36.0,2349.0,528.0,1194.0,465.0,2.5179,79000.0
16997,-124.30,41.84,17.0,2677.0,531.0,1244.0,456.0,3.0313,103600.0
16998,-124.30,41.80,19.0,2672.0,552.0,1298.0,478.0,1.9797,85800.0


In [18]:
clientes = pd.read_csv(StringIO(csv_clientes), sep=";")

In [19]:
clientes

,id_cliente,nombre,ciudad,segmento
0,C001,Ana Ruiz,Bogotá,Oro
1,C002,Luis Peña,Medellín,Plata
2,C003,Marta Gil,Cali,Oro
3,C004,Juan Ríos,Bogotá,Bronce
4,C005,Sofía Mora,Barranquilla,Plata
5,C006,Carlos Puenes,Pasto,Bronce
6,c007,Patricia Mojica,Manizales,plata
7,C008,Juan Velez,Bucaramanga,Bronce


In [20]:
print(clientes.shape)
print(clientes.head(3))

(8, 4)
  id_cliente     nombre    ciudad segmento
0       C001   Ana Ruiz    Bogotá      Oro
1       C002  Luis Peña  Medellín    Plata
2       C003  Marta Gil      Cali      Oro


In [21]:
df_california_housing.shape

(17000, 9)

In [22]:
df_california_housing.head(4)

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
0,-114.31,34.19,15.0,5612.0,1283.0,1015.0,472.0,1.4936,66900.0
1,-114.47,34.40,19.0,7650.0,1901.0,1129.0,463.0,1.8200,80100.0
2,-114.56,33.69,17.0,720.0,174.0,333.0,117.0,1.6509,85700.0
3,-114.57,33.64,14.0,1501.0,337.0,515.0,226.0,3.1917,73400.0


In [23]:
#Integrar un archivo JSON o  via API
respuesta_api = [
    {"pedido": 101, "cliente": "C001", "items": 3,
     "pago": {"medio": "Tarjeta", "valor": 150000}},
    {"pedido": 102, "cliente": "C003", "items": 1,
     "pago": {"medio": "PSE", "valor": 80000}},
    {"pedido": 103, "cliente": "C001", "items": 2,
     "pago": {"medio": "Efectivo", "valor": 45000}},
    {"pedido": 104, "cliente": "C009", "items": 5,
     "pago": {"medio": "Tarjeta", "valor": 210000}},
]
pedidos = pd.json_normalize(respuesta_api)
print(pedidos)



   pedido cliente  items pago.medio  pago.valor
0     101    C001      3    Tarjeta      150000
1     102    C003      1        PSE       80000
2     103    C001      2   Efectivo       45000
3     104    C009      5    Tarjeta      210000


# **Merge de datos**

In [24]:
#merge: INNER vs LEFT
inner = pd.merge(clientes, pedidos, left_on="id_cliente",
                 right_on="cliente", how="inner")
left = pd.merge(clientes, pedidos, left_on="id_cliente",
                right_on="cliente", how="left")

print("INNER:", inner.shape[0], "filas")
print("LEFT :", left.shape[0], "filas")
sin_compra = left[left["pedido"].isna()]["nombre"].tolist()
print("Clientes sin compras:", sin_compra)


INNER: 3 filas
LEFT : 9 filas
Clientes sin compras: ['Luis Peña', 'Juan Ríos', 'Sofía Mora', 'Carlos Puenes', 'Patricia Mojica', 'Juan Velez']


In [25]:
#Validar con indicator
outer = pd.merge(clientes, pedidos, left_on="id_cliente",
                 right_on="cliente", how="outer", indicator=True)
print(outer["_merge"].value_counts())
huerfanos = outer[outer["_merge"] == "right_only"]
print("Pedidos sin cliente:", huerfanos["pedido"].tolist())


_merge
left_only     6
both          3
right_only    1
Name: count, dtype: int64
Pedidos sin cliente: [104.0]


# **Consolidación de datos**

In [42]:
np.random.seed(1)

def ventas_mes(mes):
    return pd.DataFrame({
        "mes": mes,
        "sucursal": ["Norte", "Sur", "Centro"],
        "ventas": np.random.randint(80, 200, 3) * 1e6})

In [43]:
ene = ventas_mes("Ene")
ene

,mes,sucursal,ventas
0,Ene,Norte,117000000.0
1,Ene,Sur,187000000.0
2,Ene,Centro,92000000.0


In [44]:
feb = ventas_mes("Feb")
feb

,mes,sucursal,ventas
0,Feb,Norte,152000000.0
1,Feb,Sur,89000000.0
2,Feb,Centro,155000000.0


In [45]:
mar = ventas_mes("Mar")
mar

,mes,sucursal,ventas
0,Mar,Norte,85000000.0
1,Mar,Sur,159000000.0
2,Mar,Centro,144000000.0


In [46]:
consolidado = pd.concat([ene, feb, mar], ignore_index=True)

print(consolidado.shape)

print(consolidado.tail(3))

(9, 3)
   mes sucursal       ventas
6  Mar    Norte   85000000.0
7  Mar      Sur  159000000.0
8  Mar   Centro  144000000.0


In [47]:
consolidado

,mes,sucursal,ventas
0,Ene,Norte,117000000.0
1,Ene,Sur,187000000.0
2,Ene,Centro,92000000.0
3,Feb,Norte,152000000.0
4,Feb,Sur,89000000.0
5,Feb,Centro,155000000.0
6,Mar,Norte,85000000.0
7,Mar,Sur,159000000.0
8,Mar,Centro,144000000.0


In [48]:
# Group By con varias metricas
resumen = consolidado.groupby("sucursal").agg(
    total=("ventas", "sum"),
    promedio=("ventas", "mean"),
    meses=("mes", "count"))
resumen = (resumen / [1e6, 1e6, 1]).round(1)
print(resumen.sort_values("total", ascending=False))


          total  promedio  meses
sucursal                        
Sur       435.0     145.0    3.0
Centro    391.0     130.3    3.0
Norte     354.0     118.0    3.0


In [49]:
# Pivot y Melt
ancho = consolidado.pivot_table(index="sucursal", columns="mes",
                                values="ventas") / 1e6
ancho = ancho[["Ene", "Feb", "Mar"]]
print(ancho)

largo = ancho.reset_index().melt(id_vars="sucursal",
                                 var_name="mes", value_name="ventas_M")
print(largo.head(4))


mes         Ene    Feb    Mar
sucursal                     
Centro     92.0  155.0  144.0
Norte     117.0  152.0   85.0
Sur       187.0   89.0  159.0
  sucursal  mes  ventas_M
0   Centro  Ene      92.0
1    Norte  Ene     117.0
2      Sur  Ene     187.0
3   Centro  Feb     155.0


# **Integración SQL**

In [50]:
import sqlite3
con = sqlite3.connect(":memory:")
clientes.to_sql("clientes", con, index=False)
pedidos.rename(columns={"pago.valor": "valor"})[
    ["pedido", "cliente", "valor"]].to_sql("pedidos", con, index=False)

query = """
SELECT c.ciudad, COUNT(p.pedido) AS pedidos, SUM(p.valor) AS ventas
FROM clientes c LEFT JOIN pedidos p ON c.id_cliente = p.cliente
GROUP BY c.ciudad ORDER BY ventas DESC
"""
print(pd.read_sql(query, con))


         ciudad  pedidos    ventas
0        Bogotá        2  195000.0
1          Cali        1   80000.0
2         Pasto        0       NaN
3      Medellín        0       NaN
4     Manizales        0       NaN
5   Bucaramanga        0       NaN
6  Barranquilla        0       NaN
